# Lead Scoring MLOps — All-in-One Colab Runner

Runs the **data**, **training**, and **inference** pipelines end-to-end in a single notebook.

## Prerequisites
1. Upload the `Assignment_SreedharK` folder to your Drive root.
2. Confirmed path in Drive: `MyDrive/Assignment_SreedharK/Assignment_SreedharK/Assignment_SreedharK/`

## How to run
**Runtime → Run all** (`Ctrl+F9`). No edits, no restarts. Expected total time: ~5–7 minutes.

---
## Section 0 — Bootstrap (mount Drive, install deps, copy code)

In [ ]:
# --- Headless setup for Google Colab CLI / GitHub Actions ---
# The original notebook used drive.mount() to read the assignment folder from
# the user's Google Drive. When this notebook is run via colab-cli from GitHub
# Actions there is no interactive browser, so we instead git-clone the
# repository (containing Assignment_SreedharK.zip alongside this notebook) and
# unzip it inside the Colab VM.
#
# REPO_URL and GIT_SHA are PLACEHOLDERs that the workflow substitutes with
# https://github.com/${{ github.repository }} and ${{ github.sha }} before
# invoking `colab_cli exec`.
import os, shutil, sys, zipfile, subprocess

REPO_URL = "PLACEHOLDER_REPO_URL"
GIT_SHA  = "PLACEHOLDER_GIT_SHA"

REPO_DIR = "/content/repo"
shutil.rmtree(REPO_DIR, ignore_errors=True)
if GIT_SHA and not GIT_SHA.startswith("PLACEHOLDER"):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_SHA])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

ZIP_PATH = f"{REPO_DIR}/2_Course_continuation/_4_MLOps/Assignment_SreedharK.zip"
UNZIP_TO = "/content/Assignment_SreedharK_unzipped"
shutil.rmtree(UNZIP_TO, ignore_errors=True)
os.makedirs(UNZIP_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(UNZIP_TO)

!pip install --quiet "numpy==2.0.2" "pandas==2.2.3" "scikit-learn==1.5.2" "lightgbm==4.5.0" "mlflow==2.17.2"

SRC_ROOT = f"{UNZIP_TO}/Assignment_SreedharK/Assignment_SreedharK/Assignment_SreedharK"
assert os.path.isdir(SRC_ROOT), f"Not found: {SRC_ROOT}"

DP_SRC = f"{SRC_ROOT}/Lead_scoring_data_pipeline"
TP_SRC = f"{SRC_ROOT}/Lead_scoring_training_pipeline"
IP_SRC = f"{SRC_ROOT}/Lead_scoring_inference_pipeline"

DP_WORK = "/content/data_pipeline"
TP_WORK = "/content/training_pipeline"
IP_WORK = "/content/inference_pipeline"

for src, dst in [(DP_SRC, DP_WORK), (TP_SRC, TP_WORK), (IP_SRC, IP_WORK)]:
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)

DP_SCRIPTS = f"{DP_WORK}/AssignmentFolderFiles/scripts"
TP_SCRIPTS = f"{TP_WORK}/AssignmentFolderFiles/Scripts"
IP_SCRIPTS = f"{IP_WORK}/AssignmentFolderFiles/scripts"

print("Copied:")
print("  data       ->", DP_WORK)
print("  training   ->", TP_WORK)
print("  inference  ->", IP_WORK)


## Section 0b — Patch all `constants.py` files to local `/content/...` paths

In [2]:
# Data pipeline constants — point at local data + start with labelled CSV
dp_c = pathlib.Path(f"{DP_SCRIPTS}/constants.py")
dp_c.write_text(f"""DB_PATH = '{DP_WORK}/'
DB_FILE_NAME = 'lead_scoring_data_cleaning.db'
DATA_DIRECTORY = '{DP_WORK}/data/'
INTERACTION_MAPPING = '{DP_WORK}/mapping/'
INDEX_COLUMNS = ['created_date','city_tier','first_platform_c','first_utm_medium_c','first_utm_source_c','total_leads_droppped','referred_lead','app_complete_flag']
LEAD_SCORING_FILE='leadscoring'
""")

# Training pipeline constants — point DB_PATH at data pipeline output
tp_c = pathlib.Path(f"{TP_SCRIPTS}/constants.py")
tp_txt = tp_c.read_text()
tp_txt = tp_txt.replace("/home/airflow/dags/Lead_scoring_data_pipeline/", f"{DP_WORK}/")
tp_txt = tp_txt.replace("'/home/database/'", f"'{DP_WORK}/'")
tp_c.write_text(tp_txt)

# Inference pipeline constants — point at all 3 local dirs
ip_out = f"{IP_WORK}/output/"
os.makedirs(ip_out, exist_ok=True)
ip_c = pathlib.Path(f"{IP_SCRIPTS}/constants.py")
txt = ip_c.read_text()
txt = txt.replace("/home/airflow/dags/Lead_scoring_data_pipeline/",     f"{DP_WORK}/")
txt = txt.replace("/home/airflow/dags/Lead_scoring_inference_pipeline/", f"{IP_SCRIPTS}/")
txt = txt.replace("/home/airflow/dags/Lead_scoring_training_pipeline/",  f"{TP_WORK}/")
txt = txt.replace("'/home/database/'", f"'{DP_WORK}/'")
txt = txt.replace("'/home/Assignment/03_inference_pipeline/scripts/'", f"'{ip_out}'")
txt = txt.replace("STAGE = 'production'", "STAGE = 'Production'")
ip_c.write_text(txt)

print("--- DP constants ---");  print(dp_c.read_text())
print("--- TP constants ---");  print(tp_c.read_text())
print("--- IP constants ---");  print(ip_c.read_text())

--- DP constants ---
DB_PATH = '/content/data_pipeline/'
DB_FILE_NAME = 'lead_scoring_data_cleaning.db'
DATA_DIRECTORY = '/content/data_pipeline/data/'
INTERACTION_MAPPING = '/content/data_pipeline/mapping/'
INDEX_COLUMNS = ['created_date','city_tier','first_platform_c','first_utm_medium_c','first_utm_source_c','total_leads_droppped','referred_lead','app_complete_flag']
LEAD_SCORING_FILE='leadscoring'

--- TP constants ---
DB_PATH = '/content/data_pipeline/'
DB_FILE_NAME = 'lead_scoring_data_cleaning.db' 

DB_FILE_MLFLOW = 'Lead_scoring_mlflow_production.db'

TRACKING_URI = "http://0.0.0.0:6006"
EXPERIMENT = "Lead_Scoring_Training_Pipeline"


# model config imported from pycaret experimentation
MODEL_CONFIG = {
    'boosting_type': 'gbdt',
    'class_weight': None,
    'colsample_bytree': 1.0,
    'importance_type': 'split' ,
    'learning_rate': 0.1,
    'max_depth': -1,
    'min_child_samples': 20,
    'min_child_weight': 0.001,
    'min_split_gain': 0.0,
    'n_estimators': 100,
   

## Section 0c — Start MLflow tracking server (port 6006)

In [3]:
os.system("pkill -f 'mlflow server' 2>/dev/null; sleep 2")

proc = subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:///Lead_scoring_mlflow_production.db",
     "--host", "0.0.0.0", "--port", "6006"],
    cwd=TP_WORK,
    stdout=open("/tmp/mlflow.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    try:
        if requests.get("http://0.0.0.0:6006/").status_code == 200:
            print(f"MLflow up on http://0.0.0.0:6006 (pid {proc.pid})"); break
    except Exception:
        pass
    time.sleep(2)
else:
    print("MLflow failed — see /tmp/mlflow.log")
    !tail -30 /tmp/mlflow.log

MLflow up on http://0.0.0.0:6006 (pid 1249)


## Section 0d — Module loader helper (avoids `sys.path` conflicts between the 3 pipelines)

In [4]:
import importlib.util

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

def step(name, fn, *a, **kw):
    t = time.time()
    print(f">>> {name}")
    out = fn(*a, **kw)
    print(f"    done in {time.time()-t:.1f}s -> {out}\n")
    return out

print("helpers ready")

helpers ready


---
# Section 1 — Data Pipeline (labelled CSV)

Builds the SQLite tables: `loaded_data` → `city_tier_mapped` → `categorical_variables_mapped` → `interactions_mapped`.

In [5]:
dp_utils  = load("dp_utils",  f"{DP_WORK}/utils.py")
dp_checks = load("dp_checks", f"{DP_WORK}/data_validation_checks.py")
dp_const  = load("dp_const",  f"{DP_SCRIPTS}/constants.py")
dp_schema = load("dp_schema", f"{DP_WORK}/schema.py")
dp_city   = load("dp_city",   f"{DP_WORK}/mapping/city_tier_mapping.py")
dp_sig    = load("dp_sig",    f"{DP_WORK}/mapping/significant_categorical_level.py")

step("build_dbs",                dp_utils.build_dbs,                dp_const.DB_PATH, dp_const.DB_FILE_NAME)
step("raw_data_schema_check",    dp_checks.raw_data_schema_check,   dp_const.DATA_DIRECTORY, dp_schema.raw_data_schema)
step("load_data_into_db",        dp_utils.load_data_into_db,        dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_const.DATA_DIRECTORY, dp_const.LEAD_SCORING_FILE)
step("map_city_tier",            dp_utils.map_city_tier,            dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_city.city_tier_mapping_dict)
step("map_categorical_vars",     dp_utils.map_categorical_vars,     dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_sig.list_platform, dp_sig.list_medium, dp_sig.list_source)
step("interactions_mapping",     dp_utils.interactions_mapping,     dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_const.INTERACTION_MAPPING, dp_const.INDEX_COLUMNS)
step("model_input_schema_check", dp_checks.model_input_schema_check, dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_schema.model_input_schema)

import sqlite3, pandas as pd
cnx = sqlite3.connect(dp_const.DB_PATH + dp_const.DB_FILE_NAME)
print("Tables in DB:")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", cnx))
cnx.close()

>>> build_dbs
Creating Database
New DB Created
    done in 0.0s -> DB Created

>>> raw_data_schema_check
Raw datas schema is in line with the schema present in schema.py
    done in 0.8s -> None

>>> load_data_into_db
    done in 8.6s -> Writing to DataBase loaded_data Done or Data Already was in Table. Check Logs.

>>> map_city_tier
    done in 8.7s -> Writing to DataBase city_tier_mapped Done or Data Already was in Table. Check Logs.

>>> map_categorical_vars


/content/data_pipeline/utils.py:189: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['first_platform_c'] = "others"
/content/data_pipeline/utils.py:194: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['first_utm_medium_c'] = "others"
/content/data_pipeline/utils.py:199: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable

    done in 10.2s -> Writing to DataBase categorical_variables_mapped Done Check Logs.

>>> interactions_mapping


/content/data_pipeline/utils.py:256: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unpivot['interaction_value'] = df_unpivot['interaction_value'].fillna(0)


    done in 24.6s -> Writing to DataBase- interactions_mapped Done . Check Logs.

>>> model_input_schema_check
Models input schema is in line with the schema present in schema.py
    done in 1.2s -> None

Tables in DB:
                           name
0                   loaded_data
1              city_tier_mapped
2  categorical_variables_mapped
3           interactions_mapped


---
# Section 2 — Training Pipeline

`encode_features` → `get_trained_model` (LightGBM logged to MLflow).

In [6]:
tp_utils  = load("tp_utils",  f"{TP_SCRIPTS}/utils.py")
tp_const  = load("tp_const",  f"{TP_SCRIPTS}/constants.py")

step("encode_features",   tp_utils.encode_features,
     tp_const.DB_PATH, tp_const.DB_FILE_NAME, tp_const.ONE_HOT_ENCODED_FEATURES, tp_const.FEATURES_TO_ENCODE)

step("get_trained_model", tp_utils.get_trained_model,
     tp_const.DB_PATH, tp_const.DB_FILE_NAME, tp_const.MODEL_CONFIG, tp_const.EXPERIMENT, tp_const.TRACKING_URI)

>>> encode_features
Data has been extracted successfully from lead_scoring_model_experimentation.


/content/training_pipeline/AssignmentFolderFiles/Scripts/utils.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encoded_df.fillna(0, inplace=True)


input_data has been saved successfully to table features
input_data has been saved successfully to table target
input data has been written successfully to tables features & target
    done in 2.8s -> None

>>> get_trained_model


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Warning] Unknown parameter: silent
[LightGBM] [Info] Number of positive: 83565, number of negative: 83121
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002077 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 59
[LightGBM] [Info] Number of data points in the train set: 166686, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501332 -> initscore=0.005327
[LightGBM] [Info] Start training from score 0.005327


2026/06/15 04:48:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'LightGBM'.
2026/06/15 04:48:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM, version 1
Created version '1' of model 'LightGBM'.


[LightGBM] [Warning] Unknown parameter: silent


2026/06/15 04:48:21 INFO mlflow.tracking._tracking_service.client: 🏃 View run Lead_Scoring_Training_Pipeline at: http://0.0.0.0:6006/#/experiments/0/runs/736b79c09e034db5b360c95de352fa3c.
2026/06/15 04:48:21 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://0.0.0.0:6006/#/experiments/0.


get_trained_model has been executed successfully.
    done in 13.4s -> None



## Section 2b — Promote latest model version to **Production**

In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=tp_const.TRACKING_URI)
versions = client.search_model_versions("name='LightGBM'")
latest = max(versions, key=lambda v: int(v.version))

client.transition_model_version_stage(
    name="LightGBM",
    version=latest.version,
    stage="Production",
    archive_existing_versions=True,
)
print(f"Promoted LightGBM v{latest.version} -> Production")

Promoted LightGBM v1 -> Production


/tmp/ipykernel_746/1700877425.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


---
# Section 3 — Rebuild data DB from the **inference** CSV

The inference pipeline needs the same `interactions_mapped` table, but built from the unlabelled file.

In [8]:
# Switch data pipeline to the inference CSV
dp_c.write_text(dp_c.read_text().replace("'leadscoring'", "'leadscoring_inference_final_v2'"))
if os.path.exists(dp_const.DB_PATH + dp_const.DB_FILE_NAME):
    os.remove(dp_const.DB_PATH + dp_const.DB_FILE_NAME)
    print("Removed old DB")

# Reload constants with new value
dp_const = load("dp_const", f"{DP_SCRIPTS}/constants.py")
print("LEAD_SCORING_FILE =", dp_const.LEAD_SCORING_FILE)

# Rebuild the data tables (skip schema checks — they only apply to labelled data)
step("build_dbs",            dp_utils.build_dbs,            dp_const.DB_PATH, dp_const.DB_FILE_NAME)
step("load_data_into_db",    dp_utils.load_data_into_db,    dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_const.DATA_DIRECTORY, dp_const.LEAD_SCORING_FILE)
step("map_city_tier",        dp_utils.map_city_tier,        dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_city.city_tier_mapping_dict)

step("map_categorical_vars", dp_utils.map_categorical_vars, dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_sig.list_platform, dp_sig.list_medium, dp_sig.list_source)
inference_index_columns = [c for c in dp_const.INDEX_COLUMNS if c != 'app_complete_flag']
step("interactions_mapping", dp_utils.interactions_mapping, dp_const.DB_PATH, dp_const.DB_FILE_NAME, dp_const.INTERACTION_MAPPING, inference_index_columns)
print("Inference data ready.")

Removed old DB
LEAD_SCORING_FILE = leadscoring_inference_final_v2
>>> build_dbs
Creating Database
New DB Created
    done in 0.0s -> DB Created

>>> load_data_into_db
    done in 0.7s -> Writing to DataBase loaded_data Done or Data Already was in Table. Check Logs.

>>> map_city_tier
    done in 0.8s -> Writing to DataBase city_tier_mapped Done or Data Already was in Table. Check Logs.

>>> map_categorical_vars


/content/data_pipeline/utils.py:189: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['first_platform_c'] = "others"
/content/data_pipeline/utils.py:194: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['first_utm_medium_c'] = "others"
/content/data_pipeline/utils.py:199: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable

    done in 1.6s -> Writing to DataBase categorical_variables_mapped Done Check Logs.

>>> interactions_mapping


/content/data_pipeline/utils.py:256: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unpivot['interaction_value'] = df_unpivot['interaction_value'].fillna(0)


    done in 1.7s -> Writing to DataBase- interactions_mapped Done . Check Logs.

Inference data ready.


---
# Section 4 — Inference Pipeline

`encode_features` → `input_features_check` → `get_models_prediction` → `prediction_ratio_check`.

In [9]:
ip_utils = load("ip_utils", f"{IP_SCRIPTS}/utils.py")
ip_const = load("ip_const", f"{IP_SCRIPTS}/constants.py")

step("encode_features",       ip_utils.encode_features,
     ip_const.DB_PATH, ip_const.DB_FILE_NAME, ip_const.ONE_HOT_ENCODED_FEATURES, ip_const.FEATURES_TO_ENCODE)

step("input_features_check",  ip_utils.input_features_check,
     ip_const.DB_PATH, ip_const.DB_FILE_NAME, ip_const.ONE_HOT_ENCODED_FEATURES)

step("get_models_prediction", ip_utils.get_models_prediction,
     ip_const.DB_PATH, ip_const.DB_FILE_NAME, ip_const.MODEL_NAME, ip_const.STAGE, ip_const.TRACKING_URI)

step("prediction_ratio_check", ip_utils.prediction_ratio_check,
     ip_const.DB_PATH, ip_const.DB_FILE_NAME, ip_const.SCRIPTS_OUTPUT)

>>> encode_features
Data has been extracted successfully from lead_scoring_model_experimentation.


/content/inference_pipeline/AssignmentFolderFiles/scripts/utils.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encoded_df.fillna(0, inplace=True)


input_data has been saved successfully to table features
    done in 0.2s -> None

>>> input_features_check
Data has been extracted successfully from lead_scoring_model_experimentation.
Some of the models inputs are missing
    done in 0.1s -> None

>>> get_models_prediction
Data has been extracted successfully from lead_scoring_model_experimentation.


/usr/local/lib/python3.12/dist-packages/mlflow/store/artifact/utils/models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


[LightGBM] [Warning] Unknown parameter: silent
[1 1 1 ... 1 1 0]
input_data has been saved successfully to table predicted_output
    done in 0.5s -> Predictions are done and save in Final_Predictions Table

>>> prediction_ratio_check
Data has been extracted successfully from lead_scoring_model_experimentation.
Output file has been generated successfully /content/inference_pipeline/output/prediction_distribution_20260615044827.txt
    done in 0.0s -> None



---
# Section 5 — Results

In [10]:
import glob

cnx = sqlite3.connect(ip_const.DB_PATH + ip_const.DB_FILE_NAME)
preds    = pd.read_sql("SELECT * FROM predicted_output", cnx)
features = pd.read_sql("SELECT * FROM features", cnx)
cnx.close()

print("Total predictions:", len(preds))
print(preds["predicted_output"].value_counts(normalize=True).rename("share"))

print("\nLatest prediction distribution files:")
for f in sorted(glob.glob(ip_const.SCRIPTS_OUTPUT + "prediction_distribution_*.txt"))[-3:]:
    print("-", f)
    print(open(f).read())

# === Consolidated results CSV (features + prediction per lead) ===
results_path = f"{IP_WORK}/output/lead_scoring_predictions.csv"
results = pd.concat([features.reset_index(drop=True), preds.reset_index(drop=True)], axis=1)
results.to_csv(results_path, index=False)
print(f"\nWrote {len(results)} rows -> {results_path}")
print(results.head())

Total predictions: 24885
predicted_output
1    0.566405
0    0.433595
Name: share, dtype: float64

Latest prediction distribution files:
- /content/inference_pipeline/output/prediction_distribution_20260615044827.txt
0	43.36
1	56.64


Wrote 24885 rows -> /content/inference_pipeline/output/lead_scoring_predictions.csv
   index  first_platform_c  first_utm_medium_c  first_utm_source_c  \
0      0                 0                   0                   0   
1      1                 0                   0                   0   
2      2                 0                   0                   0   
3      3                 0                   0                   0   
4      4                 0                   0                   0   

   total_leads_droppped  city_tier  first_platform_c_Level8  \
0                   2.0        3.0                        0   
1                   1.0        1.0                        0   
2                   3.0        1.0                        0   
3       

In [ ]:
# --- Headless artifact export (replaces interactive files.download) ---
import os, shutil
os.makedirs('outputs', exist_ok=True)
shutil.copy(results_path, 'outputs/')
print('Copied', results_path, '-> outputs/')


In [12]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-06-15 10:18:27


In [ ]:
# --- gcolab wrapper: list files in outputs/ ---
import os, datetime
_out_dir = 'outputs'
os.makedirs(_out_dir, exist_ok=True)
print(f"Files in {_out_dir}:")
_files = []
for _fn in os.listdir(_out_dir):
    _fp = os.path.join(_out_dir, _fn)
    if os.path.isfile(_fp):
        _st = os.stat(_fp)
        _files.append((datetime.datetime.fromtimestamp(_st.st_mtime).strftime('%Y-%m-%d %H:%M:%S'), _st.st_size, _fn))
_files.sort()
for _m, _s, _n in _files:
    if _s < 1024:
        _h = f"{_s} B"
    elif _s < 1024*1024:
        _h = f"{_s/1024:.2f} KB"
    else:
        _h = f"{_s/(1024*1024):.2f} MB"
    print(f"{_m}  {_h:>10}  {_n}")


In [ ]:
# --- gcolab wrapper: zip outputs/ -> outputs.zip ---
import os, shutil
os.makedirs('outputs', exist_ok=True)
shutil.make_archive('outputs', 'zip', 'outputs')
print('Created outputs.zip')
